# CO₂ story — one graph per cell (upload edition)

**How to use:**
1. Run **Setup 1** (installs libraries) and **Setup 2** (defines the plotting functions).
2. Run **Setup 3** and, when prompted, **upload these 3 files**:
   `co2_per_capita.csv`, `percapita_co2_by_source.csv`, `us_co2_by_fuel.csv`
3. Then run any graph cell below — **each cell draws exactly one graph.**

## Setup 1 — install libraries

In [ ]:
!pip -q install matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('libraries ready ✓')

## Setup 2 — install the `viz_lib` library from GitHub
Installs the package straight from the repo (dependencies already came from Setup 1, so `--no-deps` keeps it fast).

In [ ]:
!pip install -q --force-reinstall --no-deps "git+https://github.com/rinikhaneja/visualisation-lib.git@claude/viz-lib-plot-requirements-dbjbc0"

import viz_lib
from viz_lib import ranked_bar, stacked_bar, stacked_area
from viz_lib.theme import apply_theme, series_color
apply_theme()
print('viz_lib installed from GitHub ✓')

## Setup 3 — upload the 3 CSV files
Run this cell, click **Choose Files**, and select the three CSVs.

In [ ]:
from google.colab import files
print('Upload: co2_per_capita.csv, percapita_co2_by_source.csv, us_co2_by_fuel.csv')
uploaded = files.upload()
print('uploaded:', list(uploaded))

## Graph 1 — Oil producers: CO₂ per person
Uses `co2_per_capita.csv`.

In [ ]:
df = pd.read_csv('co2_per_capita.csv')
df = df[df.Year.isin([2014, 2024])].pivot_table(
        index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()   # shared color scale
world = df.loc[df.Entity == 'World', 'y2024'].iloc[0]

ranked_bar(df[df.Entity.isin(OIL)], category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t', reference=world, reference_label='World average',
           title='A few small, oil-rich nations emit the most CO₂ per person',
           subtitle='Tonnes of CO₂ per person, 2024')
plt.show()

## Graph 2 — Major economies: CO₂ per person
Uses `co2_per_capita.csv` (same color scale as Graph 1).

In [ ]:
df = pd.read_csv('co2_per_capita.csv')
df = df[df.Year.isin([2014, 2024])].pivot_table(
        index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()

ranked_bar(df[df.Entity.isin(ECON)], category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t',
           title='Among big economies, the US still emits the most per person',
           subtitle='Tonnes of CO₂ per person, 2024 — same scale as the oil producers')
plt.show()

## Graph 3 — Per capita CO₂ by source, 2024
Uses `percapita_co2_by_source.csv` (a replica of the Our World in Data chart).

In [ ]:
d = pd.read_csv('percapita_co2_by_source.csv')
SEG = ['Coal','Oil','Gas','Flaring','Cement','Other industry']
OWID = {'Coal':'#6d6e70','Oil':'#c14b62','Gas':'#8c6bb1',
        'Flaring':'#c8a45c','Cement':'#2f8e7f','Other industry':'#6d8fc5'}
tonnes = lambda v: f'{v:.0f} t' if v >= 10 else f'{v:.1f} t'

stacked_bar(d, category='Entity', segments=SEG, colors=OWID,
            value_fmt=tonnes, seg_label_min=0.05,
            title='Per capita CO₂ emissions by source, 2024',
            figsize=(11, 8))
plt.show()

## Graph 4 — US CO₂ by fuel over time (the hero)
Uses `us_co2_by_fuel.csv`.

In [ ]:
h = pd.read_csv('us_co2_by_fuel.csv')
FUELS = ['Coal','Oil','Gas','Cement','Flaring','Other industry']
for f in FUELS:
    h[f] = pd.to_numeric(h[f], errors='coerce') / 1e9   # tonnes -> billion tonnes

EVENTS = [{'year':1932,'label':'1932\nGreat Depression','y':0.42},
          {'year':1945,'label':'1945\nWWII','y':0.72},
          {'year':1973,'label':'1973\nOil shock','y':0.9},
          {'year':2007,'label':'2007\nemissions peak','y':0.98},
          {'year':2020,'label':'2020\nCOVID','y':0.62}]

stacked_area(h, x='Year', series=FUELS, y_label='Billion tonnes CO₂ / year',
             title='Coal gave way to oil and gas',
             subtitle='US CO₂ emissions by fuel or industry, 1800–2024',
             events=EVENTS)
plt.show()

---
*Each graph is drawn by a `viz_lib` function defined in Setup 2, from the CSV you uploaded in Setup 3.*